In [2]:
import os
import glob
import pandas as pd
from sklearn.model_selection import train_test_split

# Load Dataset

DATA_DIR = "donateacry_corpus"

records = []

# Loop through each category folder
for category in os.listdir(DATA_DIR):

    # Create the path of the category folder
    cat_folder = os.path.join(DATA_DIR, category)
    
    if os.path.isdir(cat_folder): # Check if the path is a folder
        wav_files = glob.glob(os.path.join(cat_folder, "*.wav")) # Get all WAV audio files inside the folder
        
        for file_path in wav_files: 
            records.append({"file_path": file_path, "label": category}) # Store the file path and its label

df = pd.DataFrame(records) # Convert the records list into a DataFrame

print(df["label"].value_counts()) # Display the number of samples in each class

# Split dataset into training and testing
train_df, test_df = train_test_split( df, test_size=0.20, random_state=12, stratify=df["label"] )

print("\n(Train Set: 80%) ")
print(train_df["label"].value_counts())

print("\n(Test Set: 20%)")
print(test_df["label"].value_counts())

label
hungry        382
discomfort     27
tired          24
Name: count, dtype: int64

(Train Set: 80%) 
label
hungry        305
discomfort     22
tired          19
Name: count, dtype: int64

(Test Set: 20%)
label
hungry        77
tired          5
discomfort     5
Name: count, dtype: int64


## **VAD**

In [5]:
import numpy as np
import webrtcvad
import librosa

vad = webrtcvad.Vad(2)

SUPPORTED_SR = (8000, 16000, 32000, 48000)


def VAD(y, sr, fd=30, pad_frames=1):
  
    if fd not in (10, 20, 30):
        raise ValueError("fd must be 10, 20, or 30 ms")

    if sr not in SUPPORTED_SR:
        raise ValueError(
            f"webrtcvad only supports sample rates {SUPPORTED_SR}, got {sr}"
        )

    F_L = int(sr * fd / 1000) ##full length
    H_L = F_L // 2            ##half length

    y_int16 = np.clip(y * 32767, -32768, 32767).astype(np.int16)

    if len(y_int16) < F_L:
        # Audio shorter than a single frame —> nothing to analyze
        return np.array([])

    frames = librosa.util.frame(y_int16, frame_length=F_L, hop_length=H_L)

    vad_flags = [vad.is_speech(frame.tobytes(), sr) for frame in frames.T]

    # Find contiguous runs of speech frames (indices, inclusive) 
    runs = []
    n = len(vad_flags)
    i = 0
    while i < n:
        if vad_flags[i]:
            j = i
            while j + 1 < n and vad_flags[j + 1]:
                j += 1
            runs.append((i, j))
            i = j + 1
        else:
            i += 1

    if not runs:
        return np.array([])

    # Slice the original audio ONCE per run (no overlap duplication) ---
    cry_y = []
    pad = pad_frames * H_L
    for start_f, end_f in runs:
        start_sample = max(start_f * H_L - pad, 0)
        end_sample = min(end_f * H_L + F_L + pad, len(y))
        cry_y.append(y[start_sample:end_sample])

    return np.concatenate(cry_y)

## **Prepare**

In [6]:
from scipy.signal import butter, butter, sosfilt

def bandpass_filter(y, sr, lowcut=300.0, highcut=600.0, order=5):
    nyquist = 0.5 * sr
    low = lowcut / nyquist
    high = highcut / nyquist

    if high >= 1.0:
        raise ValueError(f"Highcut ({highcut}Hz) must be less than Nyquist frequency ({nyquist}Hz)")

    sos = butter(order, [low, high], btype='band', output='sos') # Second-Order Sections

    filtered_y = sosfilt(sos, y)
    return filtered_y.astype(np.float32)

def make_audio_5_seconds(y, sr, target_duration=5):
    target_length = int(target_duration * sr)

    y_fixed = librosa.util.fix_length(y, size=target_length)
    
    return y_fixed



## **Features Extraction**

In [ ]:
import numpy as np 
import librosa 

 
def extract_features(y, sr):

    # Extract MFCC
    mfcc = librosa.feature.mfcc(y=y, sr=sr, n_mfcc=20, n_fft=1024, n_mels=20, fmin=300, fmax=600, center=True)
    mfcc_mean = np.mean(mfcc, axis=1) 
    mfcc_std = np.std(mfcc, axis=1)  

    rms = librosa.feature.rms(y=y)
    rms_mean = np.mean(rms)
    rms_std = np.std(rms)    

    zcr = librosa.feature.zero_crossing_rate(y=y)
    zcr_mean = np.mean(zcr)
    zcr_std = np.std(zcr)   

    f0 = librosa.yin(y, fmin=300, fmax=600, sr=sr)
    f0 = f0[np.isfinite(f0)]  # Remove invalid F0 values
    if len(f0) > 0:
        f0_mean = np.mean(f0)
        f0_std = np.std(f0)
        f0_min = np.min(f0)
        f0_max = np.max(f0)
    else:
        f0_mean = 0
        f0_std = 0
        f0_min = 0
        f0_max = 0  

    #centroid = librosa.feature.spectral_centroid(y=y, sr=sr)

    #bandwidth = librosa.feature.spectral_bandwidth(y=y, sr=sr)

    #rolloff = librosa.feature.spectral_rolloff(y=y, sr=sr)
    

    #mel_spectro= librosa.feature.melspectrogram(y=y, sr=sr, n_fft=256, hop_length=128, win_length=256, window='hann', power=2.0) 
    #mel_mean = np.mean(mel_spectro, axis=1)
    #mel_std = np.std(mel_spectro, axis=1) 

    #chroma = librosa.feature.chroma_stft(y=y,sr=sr)
    #chroma_mean = np.mean(chroma, axis=1)
    #chroma_std = np.std(chroma, axis=1) 

    '''return np.hstack([ mfcc_mean, mfcc_std,
                    [rms_mean, rms_std], 
                    [zcr_mean, zcr_std],  
                    [f0_mean, f0_std], 
                    #[np.mean(centroid), np.std(centroid)], 
                    #[np.mean(bandwidth), np.std(bandwidth)], 
                    [np.mean(rolloff), np.std(rolloff)]
                ]) '''
 
    return np.hstack([mfcc_mean, mfcc_std, rms_mean, rms_std, zcr_mean, zcr_std, f0_mean, f0_std, f0_min, f0_max])    

##  **Audio Data Augmentation** 

In [8]:
import random
from audiomentations import (
    Compose,
    OneOf,
    PitchShift,
    TimeStretch,
    AddGaussianNoise, 
    HighPassFilter,
    Shift
)

def set_seed(seed=42): 
    random.seed(seed)
    np.random.seed(seed)

'''def augment_audio0(y, sr):
    y = y.astype(np.float32)
    augment = Compose([OneOf([
        PitchShift(min_semitones=-2, max_semitones=2, p=1.0),
        TimeStretch(min_rate=0.9, max_rate=1.1, leave_length_unchanged=True, p=1.0),
        Shift(min_shift=-0.5, max_shift=0.5, p=1.0),
        AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.015, p=1.0),
        Compose([
            PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
            TimeStretch(min_rate=0.9, max_rate=1.1, leave_length_unchanged=True, p=0.5),
            Shift(min_shift=-0.5, max_shift=0.5, p=0.5),
            AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.015)], shuffle=True, p=0.5)
        
    ])])

    return augment(samples=y, sample_rate=sr)
'''

def augment_audio1(y, sr):
    y = y.astype(np.float32)
    augment = Compose([
        PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
        TimeStretch(min_rate=0.9, max_rate=1.1, leave_length_unchanged=True, p=0.5),
        Shift(min_shift=-0.5, max_shift=0.5, p=0.5),
        AddGaussianNoise(min_amplitude=0.01, max_amplitude=0.015, p=0.5),
        HighPassFilter(min_cutoff_freq=3000, max_cutoff_freq=4000, p=0.3),
        
    ])

    return augment(samples=y, sample_rate=sr)



## *Extract features for training set with augmentation on minority classes*

In [9]:
# 2. Extract features for training set with augmentation on minority classes
X_train = []
y_train = []

set_seed(44)
sampling_rate=16000
for _, row in train_df.iterrows():
    y, sr = librosa.load(row['file_path'], sr=sampling_rate, duration=5.0)

    y=VAD(y, sr, 20)
    bandpass_filter(y, sr)
    make_audio_5_seconds(y, sr)
    
    # Extract features from original audio
    X_train.append(extract_features(y, sr))

    y_train.append(row['label'])

    # Augment minority classes only 
    if row['label'] == 'tired':
        for _ in range(15):
            y_aug = augment_audio1(y, sr)
            X_train.append(extract_features(y_aug, sr))
            y_train.append(row['label'])

    elif row['label'] == 'discomfort':
        for _ in range(13):
            y_aug = augment_audio1(y, sr)
            X_train.append(extract_features(y_aug, sr))
            y_train.append(row['label'])


'''
    # Augment minority classes only 
    if row['label'] != 'hungry':

        for aug_y in augment_audio(y, sr):

            X_train.append(extract_features(aug_y, sr))

            y_train.append(row['label'])
'''
X_train = np.array(X_train)

y_train = np.array(y_train)


# 3. Extract features for test set without augmentation
X_test = []

y_test = []

print("Processing test set ")

for _, row in test_df.iterrows():
    y, sr = librosa.load(row['file_path'], sr=sampling_rate)
    X_test.append(extract_features(y, sr))
    y_test.append(row['label'])

X_test = np.array(X_test)
y_test = np.array(y_test)


# 4. Print shapes and class balances
print("\nSummary ")
print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_test shape:  {y_test.shape}")

print("\nTraining set distribution after augmentation:")
print(pd.Series(y_train).value_counts())

print("\nTest set distribution:")
print(pd.Series(y_test).value_counts())

c:\Users\Catri\Desktop\Smart-Nursery\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing test set 

Summary 
X_train shape: (917, 48)
y_train shape: (917,)
X_test shape:  (87, 48)
y_test shape:  (87,)

Training set distribution after augmentation:
discomfort    308
hungry        305
tired         304
Name: count, dtype: int64

Test set distribution:
hungry        77
tired          5
discomfort     5
Name: count, dtype: int64


## **ML** ver1 (it doesn't take time)

In [21]:
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


models = {
    "dummy_most_frequent": DummyClassifier(strategy="most_frequent"),
    "svm_rbf": make_pipeline(
        StandardScaler(),
        SVC(C=2, kernel="rbf", gamma="scale", class_weight="balanced", random_state=42),
    ),
    "random_forest": RandomForestClassifier(
        n_estimators=100,
        criterion="entropy",
        max_features="sqrt",
        min_samples_leaf=2,
        min_samples_split=2,
        max_depth=10,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
    "extra_trees": ExtraTreesClassifier(
        n_estimators=130,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=42,
        class_weight="balanced",
        n_jobs=-1
    ),
}

results = []
best_name = None
best_score = -1
best_model = None

for name, candidate in models.items():
    
    candidate.fit(X_train, y_train)
    y_pred = candidate.predict(X_test)
    macro_f1 = f1_score(y_test, y_pred, average="macro")
    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    results.append({"model": name, "macro_f1": macro_f1, "balanced_accuracy": balanced_acc})
    
    if macro_f1 > best_score:
        best_name = name
        best_score = macro_f1
        best_model = candidate

results_df = pd.DataFrame(results).sort_values("macro_f1", ascending=False)
print(results_df)

y_pred = best_model.predict(X_test)

print(f"\nBest model: {best_name}")

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred, labels=best_model.classes_))
print("Labels:", best_model.classes_)

model = best_model

y_train_pred = model.predict(X_train)

print(classification_report(y_train, y_train_pred))
print(confusion_matrix(y_train, y_train_pred, labels=best_model.classes_))


                 model  macro_f1  balanced_accuracy
1              svm_rbf  0.572401           0.632035
3          extra_trees  0.393892           0.391342
2        random_forest  0.368153           0.374026
0  dummy_most_frequent  0.036232           0.333333

Best model: svm_rbf
              precision    recall  f1-score   support

  discomfort       0.50      0.80      0.62         5
      hungry       0.95      0.90      0.92        77
       tired       0.17      0.20      0.18         5

    accuracy                           0.85        87
   macro avg       0.54      0.63      0.57        87
weighted avg       0.87      0.85      0.86        87

[[ 4  0  1]
 [ 4 69  4]
 [ 0  4  1]]
Labels: ['discomfort' 'hungry' 'tired']
              precision    recall  f1-score   support

  discomfort       0.97      0.90      0.94       308
      hungry       0.95      0.99      0.97       305
       tired       0.93      0.95      0.94       304

    accuracy                           0.95

In [15]:
# Save trained model
import joblib

joblib.dump(best_model, "best_model.pkl")

print("Model saved successfully!")

Model saved successfully!


## **ML version2** (it takes 1 hour in average) (the best)

In [13]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn.utils.parallel")

from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.metrics import balanced_accuracy_score, classification_report, confusion_matrix, f1_score, recall_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC

# 1. Random Forest-------------

RF= RandomForestClassifier(random_state=42, n_jobs=-1)

param_grid_RF = {
    "n_estimators": [100, 300, 500],
    "max_depth": [None, 10, 20],
    "max_features": ["sqrt", "log2"],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
    "class_weight": ["balanced","balanced_subsample"],
    "criterion": ["gini", "entropy"]
}

RF_search = GridSearchCV(
    estimator=RF,
    param_grid=param_grid_RF,
    cv=10,
    verbose=1,
    scoring="recall_macro",
    
)

# 2. SVM--------------------------

SVM= make_pipeline(
    StandardScaler(), 
    SVC(random_state=42, kernel="rbf")
)


param_grid_SVM = {
    "svc__C": [0.1, 1, 2, 5, 10, 100],
    "svc__gamma": ["scale", "auto", 0.001, 0.01, 0.1],
    "svc__class_weight": ["balanced", None],
    "svc__decision_function_shape":["ovr", "ovo"]
}

SVM_search = GridSearchCV(
    estimator=SVM,
    param_grid=param_grid_SVM,
    cv=10,
    verbose=1,
    scoring="recall_macro",
    n_jobs=-1
)

# 3. Extra Trees --------------------

extra_trees = ExtraTreesClassifier(
    n_estimators=130,
    max_features="sqrt",
    min_samples_leaf=2,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

# 4. Dummy baseline ------------------------

dummy = DummyClassifier(strategy="most_frequent")

# Models ---------------------------------------
models = {
    "dummy_most_frequent": dummy,
    "svm_rbf": SVM_search,
    "random_forest": RF_search,
    "extra_trees": extra_trees,
}

# 6. Train + Evaluate ---------------------------

results = []
best_name = None
best_score = -1
best_model = None

for name, candidate in models.items():

    print(f"\nTraining {name}...")
    
    candidate.fit(X_train, y_train)
    y_pred = candidate.predict(X_test)
    print(confusion_matrix(y_test, y_pred))

    macro_f1 = f1_score(y_test, y_pred, average="macro")
    recalls_per_class = recall_score(y_test, y_pred, average=None)
    macro_recall = np.mean(recalls_per_class)
    min_class_recall = np.min(recalls_per_class)

    balanced_acc = balanced_accuracy_score(y_test, y_pred)
    results.append({"model": name, "macro_recall": macro_recall, "min_class_recall": min_class_recall,})
    
    if macro_recall > best_score:
        best_name = name
        best_score = macro_recall
        best_model = candidate


# Results ------------------------

results_df = pd.DataFrame(results).sort_values("macro_recall", ascending=False)
print(results_df)

# Best model ------------------------------------------------------

y_pred = best_model.predict(X_test)

print(f"\nBest model: {best_name}")

print(classification_report(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))
#print("Labels:", best_model.classes_)

model = best_model

y_train_pred = model.predict(X_train)

# test classification report ------------------------------------

print(classification_report(y_train, y_train_pred))

# Test confusion matrix -----------------------------------------
print(confusion_matrix(y_train, y_train_pred))

print("\nLabels:")
print(best_model.classes_)

# Training performance -------------------------------------------

y_train_pred = best_model.predict(X_train)

print("\nTRAIN classification report:")

print(
    classification_report(
        y_train,
        y_train_pred
    )
)

print("\nTRAIN confusion matrix:")

print(
    confusion_matrix(
        y_train,
        y_train_pred,
        labels=best_model.classes_
    )
)


Training dummy_most_frequent...
[[ 5  0  0]
 [77  0  0]
 [ 5  0  0]]

Training svm_rbf...
Fitting 10 folds for each of 120 candidates, totalling 1200 fits
[[ 4  0  1]
 [ 5 68  4]
 [ 0  4  1]]

Training random_forest...
Fitting 10 folds for each of 288 candidates, totalling 2880 fits
[[ 2  3  0]
 [ 4 72  1]
 [ 0  5  0]]

Training extra_trees...
[[ 1  4  0]
 [ 3 74  0]
 [ 0  5  0]]
                 model  macro_recall  min_class_recall
1              svm_rbf      0.627706               0.2
2        random_forest      0.445022               0.0
3          extra_trees      0.387013               0.0
0  dummy_most_frequent      0.333333               0.0

Best model: svm_rbf
              precision    recall  f1-score   support

  discomfort       0.44      0.80      0.57         5
      hungry       0.94      0.88      0.91        77
       tired       0.17      0.20      0.18         5

    accuracy                           0.84        87
   macro avg       0.52      0.63      0.56     